# Qwen3.6-27B Claude Opus Reasoning Distill v2 INT4 AutoRound Cookbook

This notebook is a practical starting point for building with CoreWorxLab/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2-int4-AutoRound.

What is covered:
- local setup for a self-hosted checkpoint
- basic OpenAI-compatible chat completions against a vLLM server
- streaming reasoning output
- framework integration with LangChain
- tool-enabled agent loop
- long-context QA pattern
- speculative decoding with Qwen3 MTP
- quantization details that matter for deployment

Source: https://huggingface.co/CoreWorxLab/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2-int4-AutoRound

## 0) Setup and Installation

In [ ]:
# Run this once in a fresh environment.
# %pip install -U openai transformers accelerate langchain langchain-openai tiktoken

In [ ]:
import json
import os
from typing import Any, cast
from openai import OpenAI

MODEL = "CoreWorxLab/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2-int4-AutoRound"
BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:8000/v1")
API_KEY: Any = os.getenv("OPENAI_API_KEY") or "EMPTY"

client = OpenAI(api_key=lambda: API_KEY, base_url=BASE_URL)

print(f"Using model: {MODEL}")
print(f"Base URL: {BASE_URL}")
print("Tip: start vLLM with reasoning-parser qwen3 before running the examples below.")

## 1) Basic Usage with Provider SDK

In [ ]:
messages: list[Any] = [
    {"role": "system", "content": "You are a concise coding assistant."},
    {"role": "user", "content": "List practical ways an INT4 reasoning model changes deployment planning for a small team."},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
)
print(response.choices[0].message.content)

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Explain why AutoRound INT4 can be useful for running a large reasoning model on a single GPU."}],
    stream=True,
)

is_answering = False
print("\n" + "=" * 20 + " Reasoning " + "=" * 20)
for chunk in stream:
    delta = chunk.choices[0].delta
    reasoning = getattr(delta, "reasoning_content", None)
    content = getattr(delta, "content", None)

    if reasoning and not is_answering:
        print(reasoning, end="", flush=True)

    if content:
        if not is_answering:
            is_answering = True
            print("\n" + "=" * 20 + " Answer " + "=" * 20)
        print(content, end="", flush=True)
print()

### Serve with vLLM

The model card shows a vLLM launch with `reasoning-parser qwen3`. That is the unique part to keep intact if you want reasoning traces and OpenAI-compatible responses locally.

```bash
vllm serve CoreWorxLab/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2-int4-AutoRound \
    --tensor-parallel-size 1 \
    --max-model-len 262144 \
    --gpu-memory-utilization 0.95 \
    --reasoning-parser qwen3
```

With speculative decoding enabled:

```bash
vllm serve CoreWorxLab/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2-int4-AutoRound \
    --tensor-parallel-size 1 \
    --max-model-len 262144 \
    --gpu-memory-utilization 0.95 \
    --reasoning-parser qwen3 \
    --speculative-config '{"method":"qwen3_next_mtp","num_speculative_tokens":2}'
```

## 2) Framework Integration

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model=MODEL,
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0,
)

msg = llm.invoke("Write a rollout checklist for moving a reasoning model behind an internal API.")
print(msg.content)

## 3) Building Agents

In [ ]:
def check_gpu_budget(gpu_memory_gb: int):
    if gpu_memory_gb >= 24:
        status = "comfortable"
    elif gpu_memory_gb >= 16:
        status = "tight but workable"
    else:
        status = "likely too small for this checkpoint"
    return {"gpu_memory_gb": gpu_memory_gb, "status": status}

tools = [
    {
        "type": "function",
        "function": {
            "name": "check_gpu_budget",
            "description": "Estimate whether a GPU budget is enough for a local reasoning model deployment.",
            "parameters": {
                "type": "object",
                "properties": {"gpu_memory_gb": {"type": "integer"}},
                "required": ["gpu_memory_gb"],
            },
        },
    }
]

In [ ]:
def check_gpu_budget(gpu_memory_gb: int):
    if gpu_memory_gb >= 24:
        status = "comfortable"
    elif gpu_memory_gb >= 16:
        status = "tight but workable"
    else:
        status = "likely too small for this checkpoint"
    return {"gpu_memory_gb": gpu_memory_gb, "status": status}

tools: list[Any] = [
    {
        "type": "function",
        "function": {
            "name": "check_gpu_budget",
            "description": "Estimate whether a GPU budget is enough for a local reasoning model deployment.",
            "parameters": {
                "type": "object",
                "properties": {"gpu_memory_gb": {"type": "integer"}},
                "required": ["gpu_memory_gb"],
            },
        },
    }
]

## 4) Advanced Applications

In [ ]:
def chunk_text(text: str, chunk_size: int = 3000):
    return [text[i:i + chunk_size] for i in range(0, len(text), chunk_size)]

# Replace this stub with your actual long document or codebase excerpt.
long_doc = """
This model is optimized for self-hosted reasoning workloads where memory footprint matters.
INT4 AutoRound reduces the deployment cost relative to a full precision checkpoint.
The vLLM launch path keeps the model compatible with OpenAI-style clients.
""" * 300

chunks = chunk_text(long_doc)
context = "\n\n".join(chunks[:30])

qa = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer using only the provided context."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: What is the main deployment tradeoff this model makes?"},
    ],
)
print(qa.choices[0].message.content)

In [ ]:
# Multimodal input is not the focus of this quantized reasoning checkpoint.
# Keep this notebook centered on text reasoning, deployment, and memory-efficient serving.
deployment_notes = {
    "base_model": "TeichAI/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2",
    "quantization": "INT4 symmetric, group_size=128 (W4A16)",
    "algorithm": "AutoRound",
    "compatibility": ["vLLM", "SGLang", "compressed-tensors"],
    "context_len": 262144,
}

for key, value in deployment_notes.items():
    print(f"{key}: {value}")

## Model-Specific Notes

- This is a quantized release of TeichAI/Qwen3.6-27B-Claude-Opus-Reasoning-Distill-v2.
- The unique deployment angle is INT4 AutoRound, symmetric group-size 128, which is the part that reduces memory pressure.
- The model card shows vLLM support with `--reasoning-parser qwen3`.
- The model card also shows speculative decoding with `qwen3_next_mtp`.
- The documented max model length is 262144 tokens.
- The release is marked Apache-2.0, but the model card asks you to follow the original model license as well.
- This notebook avoids flashy multimodal examples because the differentiator here is self-hosted reasoning throughput and deployment efficiency, not a new input modality.

## Footer

If you are comparing this checkpoint with the full-precision model, the practical question is whether lower VRAM and easier serving are worth the quantization tradeoff for your task. Start with the vLLM path, confirm reasoning output quality on your own prompts, then decide whether speculative decoding is a net win for your latency target.